# Plot heatmap 3 protected groups

### alpha minority2 \in [0,1] is fixed

### axes x: alpha majority

### axes y: alpha minority1

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from typing import Optional, Tuple, List
import os

In [2]:
def create_path(path):
    if not os.path.exists(path):
        os.makedirs(path)

def make_pivot(df: pd.DataFrame, fm2_val: float, metric: str,
               col_param: str, row_param: str) -> pd.DataFrame:
    """Filtra per fair_minority2 e costruisce la pivot table."""
    sub = df[np.isclose(df["fair_minority2"], fm2_val)]
    mean_col = f"{metric}_mean"
    pivot = sub.pivot(index=row_param, columns=col_param, values=mean_col)
    return pivot.sort_index(ascending=False)


def plot_grid(
    df1: pd.DataFrame,
    df2: pd.DataFrame,
    metric: str,
    fm2_values: list,
    col_param: str = "fair_majority",
    row_param: str = "fair_minority",
    metric_label: Optional[str] = None,
    range_values: Optional[Tuple[float, float]] = None,
    alg1_name: str = "Algorithm 1",
    alg2_name: str = "Algorithm 2",
    dataset: str = "",
    sensitive: str = "",
    plot_path: str = ".",
    save_fig: bool = False,
    sensitive_degree: str = None
):
    n_rows = len(fm2_values)
    n_cols = 2                          # colonna 1 = alg1, colonna 2 = alg2

    fig, axes = plt.subplots(
        n_rows, n_cols,
        figsize=(6 * n_cols, 5 * n_rows),
        squeeze=False,
    )

    mean_col = f"{metric}_mean"
    label    = metric_label or metric

    # Calcola range globale se non fornito
    if range_values is None:
        all_vals = pd.concat([df1[mean_col], df2[mean_col]], ignore_index=True).dropna()
        vmin, vmax = all_vals.min(), all_vals.max()
    else:
        vmin, vmax = range_values

    for r, fm2 in enumerate(fm2_values):
        for c, (df, alg_name) in enumerate([(df1, alg1_name), (df2, alg2_name)]):
            ax     = axes[r][c]
            pivot  = make_pivot(df, fm2, metric, col_param, row_param)

            # Ultima colonna → mostra colorbar; altrimenti nascondila
            show_cbar = (c == n_cols - 1)

            sns.heatmap(
                pivot,
                ax=ax,
                cmap="viridis",
                vmin=vmin,
                vmax=vmax,
                annot=True,
                fmt=".2f",
                annot_kws={"size": 7},
                cbar=show_cbar,
                cbar_kws={"label": label} if show_cbar else {},
            )

            ax.set_xlabel(col_param)
            ax.set_ylabel(row_param)
            ax.set_title(
                f"{dataset} – {sensitive}\n"
                f"{alg_name}  |  fair_minority2 = {fm2:.1f}"
            )
            ax.grid(False)

    plt.tight_layout()

    if save_fig:
        if sensitive_degree is not None:
            out = f"{plot_path}/sdegree{sensitive_degree}_{metric}_heatmap.png"
        else:
            out = f"{plot_path}/{metric}_heatmap.png"
        fig.savefig(out, dpi=300, bbox_inches="tight")
        print(f"Salvato in: {out}")
    else:
        plt.show()

    plt.close(fig)

In [3]:
def plot_single(
    df: pd.DataFrame,
    metric: str,
    col_param: str = "fair_majority",
    row_param: str = "fair_minority",
    metric_label: Optional[str] = None,
    range_values: Optional[Tuple[float, float]] = None,
    alg_name: str = "Algorithm",
    dataset: str = "",
    sensitive: str = "",
    plot_path: str = ".",
    save_fig: bool = False,
    sensitive_degree: float = 1.0,
    fm2_val: float = 1.0,
):
    mean_col = f"{metric}_mean"
    label    = metric_label or metric

    # Filtra per sensitive_degree e fair_minority2
    sub = df[np.isclose(df["sensitive_p"], sensitive_degree) &
             np.isclose(df["fair_minority2"], fm2_val)]
    if sub.empty:
        raise ValueError(
            fr"Nessun dato per sensitive_degree={sensitive_degree} e $\alpha$_minority2={fm2_val}"
        )

    pivot = sub.pivot(index=row_param, columns=col_param, values=mean_col)
    pivot = pivot.sort_index(ascending=False)

    if range_values is None:
        vmin, vmax = sub[mean_col].min(), sub[mean_col].max()
    else:
        vmin, vmax = range_values

    fig, ax = plt.subplots(figsize=(6, 5))

    sns.heatmap(
        pivot,
        ax=ax,
        cmap="viridis",
        vmin=vmin,
        vmax=vmax,
        annot=False,
        cbar=True,
        cbar_kws={"label": label},
    )

    ax.set_xlabel(r"$\alpha$ majority",fontsize=16)
    ax.set_ylabel(r"$\alpha$ minority1",fontsize=16)
    ax.set_title(
        f"{dataset} – {sensitive}\n"
        f"{alg_name}  |  $\alpha$ minority2 = {fm2_val:.1f}  |  sensitive_degree = {sensitive_degree:.1f}",
        fontsize=16
    )
    ax.tick_params(axis="both", labelsize=14)
    
    cbar = ax.collections[0].colorbar
    cbar.set_label(label, fontsize=16)
    cbar.ax.tick_params(labelsize=14)

    ax.grid(False)

    plt.tight_layout()

    if save_fig:
        out = f"{plot_path}/{alg_name}_sdegree{sensitive_degree}_fm2{fm2_val}_{metric}_heatmap.png"
        fig.savefig(out, dpi=300, bbox_inches="tight")
        print(f"Salvato in: {out}")
    else:
        plt.show()

    plt.close(fig)

In [4]:
root = os.getcwd()
root

'/home/peiretti/fair-clustering'

## Real-world datasets: ML age

In [5]:
dataset = "movielens-1m"
sensitive = "age"

taucc_fair = pd.read_csv(root + f"/results/{dataset}/{sensitive}/taucc_fair/init_random/aggregated.csv")
taucc_fair_max = pd.read_csv(root + f"/results/{dataset}/{sensitive}/taucc_fair_max/init_random/aggregated.csv")

path_plot = "/home/peiretti/fair-clustering/plots/movielens-1m/age/heatmap"

In [6]:
# ── Configurazione ────────────────────────────────────────────────────────────

ALG1_NAME = r"Fair-$\tau$CC v1"
ALG2_NAME = r"Fair-$\tau$CC v2"

METRIC       = "tau_y"          # cambia qui la metrica
METRIC_LABEL = "tau_y"
DATASET      = dataset
SENSITIVE    = sensitive
PLOT_PATH    = path_plot                     # cartella di output

FAIR_MINORITY2_VALUES = [0.0, 0.3, 0.5, 0.8, 1.0]
COL_PARAM = "fair_majority"
ROW_PARAM = "fair_minority"

SAVE_FIG  = False   # True → salva su disco invece di mostrare

# ─────────────────────────────────────────────────────────────────────────────

In [14]:
# ── Main ──────────────────────────────────────────────────────────────────────
if __name__ == "__main__":
    
    plot_grid(
        df1=taucc_fair,
        df2=taucc_fair_max,
        metric=METRIC,
        fm2_values=FAIR_MINORITY2_VALUES,
        col_param=COL_PARAM,
        row_param=ROW_PARAM,
        metric_label=METRIC_LABEL,
        range_values=None,              # o es. (0.0, 1.0) per range fisso
        alg1_name=ALG1_NAME,
        alg2_name=ALG2_NAME,
        dataset=DATASET,
        sensitive=SENSITIVE,
        plot_path=PLOT_PATH,
        save_fig=SAVE_FIG,
    )

KeyError: 'sensitive_p'

## Synthetic data

In [ ]:
dataset = "synthetic"
clusters = 3
groups = 3
sensitive_degree = 1.0

taucc_fair = pd.read_csv(root + f"/results/{dataset}/clus{clusters}/taucc_fair/aggregated_groups{groups}.csv").query(f"sensitive_p=={sensitive_degree}")
taucc_fair_max = pd.read_csv(root + f"/results/{dataset}/clus{clusters}/taucc_fair_max/aggregated_groups{groups}.csv").query(f"sensitive_p=={sensitive_degree}")

path_plot = f"/home/peiretti/fair-clustering/plots/synthetic/clus{clusters}/groups{groups}/heatmap"

In [ ]:
create_path(path_plot)

In [ ]:
path_plot

In [ ]:
ALG1_NAME = r"Fair-$\tau$CC v1"
ALG2_NAME = r"Fair-$\tau$CC v2"

METRIC       = "tau_y"          # cambia qui la metrica
METRIC_LABEL = "tau_y"
DATASET      = dataset
CLUS_GROUPS  = f"clus{clusters}_groups{groups}"
PLOT_PATH    = path_plot                     # cartella di output

FAIR_MINORITY2_VALUES = np.unique(taucc_fair["fair_minority2"])
COL_PARAM = "fair_majority"
ROW_PARAM = "fair_minority1"

SAVE_FIG  = True   # True → salva su disco invece di mostrare

In [8]:
metrics = {
    #("tau_x", "tau_x"),
    #("tau_y", "tau_y"),
    ("balance_bera", "Balance")
}

for METRIC, METRIC_LABEL in metrics:
    
    """
    plot_grid(
        df1=taucc_fair,
        df2=taucc_fair_max,
        metric=METRIC,
        fm2_values=FAIR_MINORITY2_VALUES,
        col_param=COL_PARAM,
        row_param=ROW_PARAM,
        metric_label=METRIC_LABEL,
        range_values=None,              # o es. (0.0, 1.0) per range fisso
        alg1_name=ALG1_NAME,
        alg2_name=ALG2_NAME,
        dataset=DATASET,
        sensitive=CLUS_GROUPS,
        plot_path=PLOT_PATH,
        save_fig=SAVE_FIG,
        sensitive_degree=sensitive_degree
    )
    """
    
    plot_single(
        df=taucc_fair,
        metric=METRIC,
        col_param=COL_PARAM,
        row_param=ROW_PARAM,
        metric_label=METRIC_LABEL,
        range_values=(0.0,1.0),
        alg_name=ALG1_NAME,
        dataset=DATASET,
        sensitive=CLUS_GROUPS,
        plot_path=PLOT_PATH,
        save_fig=SAVE_FIG,
        sensitive_degree=sensitive_degree,
        fm2_val = 1.0
    )
    
    plot_single(
        df=taucc_fair_max,
        metric=METRIC,
        col_param=COL_PARAM,
        row_param=ROW_PARAM,
        metric_label=METRIC_LABEL,
        range_values=(0.0,1.0),
        alg_name=ALG2_NAME,
        dataset=DATASET,
        sensitive=CLUS_GROUPS,
        plot_path=PLOT_PATH,
        save_fig=SAVE_FIG,
        sensitive_degree=sensitive_degree,
        fm2_val = 1.0
    )

NameError: name 'CLUS_GROUPS' is not defined